# ANN Lab 6: Backpropagation from scratch

**Name:** Muhammad Taqui
**Enrollment:** 01-136221-021
**Class:** BS-AI(6A)

Aim: implement a two-layer neural network with sigmoid activations and manual backpropagation, and train it on the XOR problem.

In [1]:
import numpy as np


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def sigmoid_derivative(x):
    return x * (1 - x)


def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


def feedforward(X, weights1, weights2, bias1, bias2):
    # Hidden layer
    z1 = np.dot(X, weights1) + bias1
    a1 = sigmoid(z1)
    # Output layer
    z2 = np.dot(a1, weights2) + bias2
    a2 = sigmoid(z2)
    return a1, a2


def backprop(X, y, weights1, weights2, bias1, bias2, a1, a2, learning_rate):
    # Output layer error
    error_output = y - a2
    delta_output = error_output * sigmoid_derivative(a2)

    # Hidden layer error
    error_hidden = delta_output.dot(weights2.T)
    delta_hidden = error_hidden * sigmoid_derivative(a1)

    weights2 += a1.T.dot(delta_output) * learning_rate
    bias2 += np.sum(delta_output, axis=0) * learning_rate
    weights1 += X.T.dot(delta_hidden) * learning_rate
    bias1 += np.sum(delta_hidden, axis=0) * learning_rate

    return weights1, weights2, bias1, bias2


## Initialize network and training data (XOR)

In [2]:
input_size = 3   # number of input features
hidden_size = 4  # neurons in the hidden layer
output_size = 1  # number of outputs

np.random.seed(42)
weights1 = np.random.rand(input_size, hidden_size)
weights2 = np.random.rand(hidden_size, output_size)
bias1 = np.random.rand(hidden_size)
bias2 = np.random.rand(output_size)

X = np.array([[0, 0, 1],
              [1, 1, 1],
              [1, 0, 1],
              [0, 1, 1]])
y = np.array([[0], [1], [1], [0]])


## Train

In [3]:
epochs = 10000
learning_rate = 0.1

for epoch in range(epochs):
    a1, a2 = feedforward(X, weights1, weights2, bias1, bias2)
    weights1, weights2, bias1, bias2 = backprop(
        X, y, weights1, weights2, bias1, bias2, a1, a2, learning_rate
    )
    if epoch % 1000 == 0:
        loss = mse_loss(y, a2)
        print(f'Epoch {epoch}, Loss: {loss}')

print("Final Output:", a2)
print(f'Final Weights1:\n{weights1}')
print(f'Final Bias1:\n{bias1}')
print(f'Final Weights2:\n{weights2}')
print(f'Final Bias2:\n{bias2}')


Epoch 0, Loss: 0.36246315741312624
Epoch 1000, Loss: 0.015838958806720024
Epoch 2000, Loss: 0.003971523964250476
Epoch 3000, Loss: 0.0021089903762651217
Epoch 4000, Loss: 0.001405170736089331
Epoch 5000, Loss: 0.0010433713935205607
Epoch 6000, Loss: 0.0008252722201057543
Epoch 7000, Loss: 0.0006802937415867795
Epoch 8000, Loss: 0.0005773256745086845
Epoch 9000, Loss: 0.0005006138596576535
Final Output: [[0.0240819 ]
 [0.97879071]
 [0.98234866]
 [0.02059874]]
Final Weights1:
[[ 4.69019574  0.85245852  3.79391067 -0.29630106]
 [-0.02153542  0.20051072 -0.02894173  0.91108001]
 [-1.02836627  0.84933767 -1.15729441  1.39322051]]
Final Bias1:
[-1.32523904  0.66602152 -0.74593389  0.7145398 ]
Final Weights2:
[[ 5.65375398]
 [-0.52717761]
 [ 4.11665386]
 [-1.98007434]]
Final Bias2:
[-2.52896704]


## 1. Introduction to neural networks

A neural network is a computational model inspired by biological neurons. It consists of layers of interconnected nodes that process input data to produce an output, learning patterns from data to perform tasks such as classification or regression.

**Key components**
- **Neurons:** the basic processing unit; each receives input, applies a function, and passes the result forward.
- **Layers:** input, hidden, and output layers.
- **Weights and biases:** connections between neurons carry weights, and a bias is added before the activation function is applied.

## 2. Feedforward neural networks

### 2.1 Structure

A feedforward neural network (FNN) is the simplest architecture: connections between nodes do not form a cycle, and information moves in one direction from input to output.

- **Input layer:** accepts input data, performs no computation.
- **Hidden layers:** multiply input by weights, add biases, and apply an activation function.
- **Output layer:** produces the final prediction.

### 2.2 Mathematical computation

For each neuron, the feedforward process computes a weighted sum

$$z = Wx + b$$

followed by an activation function

$$a = f(z)$$

where $W$ is the weight matrix, $x$ is the input, and $b$ is the bias term.

## 3. Activation functions

Activation functions introduce non-linearity; without them a network reduces to a linear model.

- **Sigmoid:** $f(z) = \dfrac{1}{1 + e^{-z}}$, maps input to $(0, 1)$.
- **ReLU:** $\text{ReLU}(z) = \max(0, z)$, replaces negative values with zero.
- **Tanh:** $\tanh(z) = \dfrac{2}{1 + e^{-2z}} - 1$, maps input to $(-1, 1)$.

## 4. Loss function

The loss function measures how well the network's output matches the target, providing the signal used to adjust weights.

- **Mean squared error (MSE):** used for regression, $\text{MSE} = \dfrac{1}{n}\sum (y_{true} - y_{pred})^2$.
- **Cross-entropy loss:** used for classification, penalizes confident wrong predictions more heavily.

## 5. Backpropagation

Backpropagation trains a network by minimizing loss through two phases:

- **Forward pass:** input passes through the network to produce an output; error is computed.
- **Backward pass:** error propagates backward and weights/biases are updated via gradient descent.

### 5.1 Gradient descent

Gradient descent minimizes the loss by moving weights in the direction opposite the gradient:

$$W_{new} = W_{old} - \eta \, \nabla_W L$$

where $\eta$ is the learning rate.

### 5.2 Steps

1. Compute error at the output using the loss function.
2. Calculate the gradient of the loss with respect to each weight.
3. Update weights and biases using gradient descent.

### 5.3 Chain rule

The chain rule computes gradients of the loss with respect to each weight by taking partial derivatives layer by layer and propagating the error backward.

## 6. Training process

1. Randomly initialize weights and biases.
2. Feedforward: compute output for a given input.
3. Compute loss against the target output.
4. Backpropagate: compute gradients of the loss with respect to weights/biases.
5. Update weights and biases using gradient descent.
6. Repeat until the loss converges or a set number of epochs is reached.

## 7. Conclusion

Building a neural network from scratch involves defining the structure (layers, weights, biases), applying activation functions, and training with backpropagation. Iteratively adjusting parameters through gradient descent lets the network approximate complex functions and improve its predictions over epochs.